<a href="https://colab.research.google.com/github/BumRushBangz/Summer26Projects/blob/main/Ch3FromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# load the csv into a table
df = pd.read_csv("sample_data/california_housing_train.csv")

# pull the predictor column (median_income) into a list called x
# pull the response column (median_house_value) into a list called y
x = df['median_income'].tolist()
y = df['median_house_value'].tolist()

# calculate mean of x and mean of y
x_mean=sum(x)/len(x)
y_mean=sum(y)/len(y)

# Numerator is gonna the sum of all: x-xmean * y-ymean. This measures the
# direction of the slope, or whether the relationship is positive
# or negative and to what degree. (sxy)
# Denominator is gonna be the sum of all: (x-xmean)^2
# BECAUSE if you were to not square it, the sum of all deviations would always
# be zero, the mean deviation is zero of course. This is simply the total
# spread of x; you would divide by n to get variance. (sxx)
# start total at 0, loop over positions, add each row's contribution
total_sxy = 0
for i in range(len(x)):
    total_sxy = total_sxy + (x[i] - x_mean) * (y[i] - y_mean)

total_sxx = 0
for i in range(len(x)):
    total_sxx = total_sxx + (x[i] - x_mean)**2

# slope b1 = sxy / sxx
# intercept b0 = y_mean - b1 * x_mean
# print both
b1 = total_sxy / total_sxx
print(b1)

b0=y_mean-b1*x_mean
print(b0)

42054.074874056714
43980.62815629426


In [2]:
# define predict_hv(income): return the line's guess for that income
# do a sanity check
def predict_hv(income):
  return b0+b1*income

predict_hv(7)

338359.15227469127

In [3]:
#RSS Is the squared error using my line - residual
#TSS Is the squared error using the mean as a dumb approximation - total
rss=0
for i in range(len(x)):
    rss = rss + (y[i] - predict_hv(x[i]))**2

tss=0
for i in range(len(x)):
    tss = tss + (y[i] - y_mean)**2

# r2 = 1 - rss/tss   -> fraction of the dumb model's error my line eliminated
r2 = 1-(rss/tss)
print('R^2 is', r2)

# validate: r = sxy / (sqrt(sxx) * sqrt(tss)); print r**2, must match r2
r = total_sxy / ((total_sxx**0.5) * (tss**0.5))
print('Validated R^2 is', r**2)

se_b1 = ((rss/(len(x)-2)) / total_sxx)**0.5
print('Standard error is', se_b1)

t=b1/se_b1
print('T value is', t)

R^2 is 0.4786849323758344
Validated R^2 is 0.4786849323758335
Standard error is 336.6157490846716
T value is 124.9319884420456


In [4]:
#Building OLS From Scratch

# Build X. Add each subsequent x value to it.
# 1, x[1] because of matrix multiplication, b0 will be multipled by 1 because
# it's the intercept.
# Easy to add on to, just add new variables after x[]. Could add rooms[i] etc.
X=[]
for i in range(len(x)):
  X.append([1,x[i]])

# Similar to X, but we just need one column (a list of one-item rows) for
# the matrix multiplication
Y_col=[]
for i in range(len(y)):
  Y_col.append([y[i]])

def transpose(M):
# Create a blank canvas.
  T = []
# outer loop HOLDS a column, inner loop works down it.
  for j in range(len(M[0])):
    new_row=[]
    for i in range(len(M)):
      new_row.append(M[i][j])
    T.append(new_row)
  return T

def dot(row, col):
# Create a blank canvas.
    total = 0
    for k in range(len(row)):
        total = total + row[k]*col[k]
# The first row value gets multiplied by the first col value, the second by
# the second and then added to the previous result, and so on.
    return total

def matmul(A,B):
# First we need B in matrix multiplication form.
  Bt = transpose(B)
# Blank canvas
  C = []
  for i in range (len(A)):
    row = []
# Now we multiply A by the transposed B, same thing as earlier with dot.
    for j in range (len(Bt)):
      #print("i =", i, "j =", j, "|", A[i], "·", Bt[j], "=", dot(A[i], Bt[j]))
      row.append(dot(A[i], Bt[j]))
    C.append(row)
  return C

print(matmul([[1, 2], [3, 4]], [[5, 6], [7, 8]]))

[[19, 22], [43, 50]]


In [5]:
def inv2x2(M):
# This is just making the inverse function so that we can use all of our matrix
# tools.
    a = M[0][0]; b = M[0][1]
    c = M[1][0]; d = M[1][1]
    det = a*d - b*c
    return [[ d/det, -b/det],
            [-c/det,  a/det]]

In [8]:
# beta = (XtX)^-1 Xt y
Xt      = transpose(X)
XtX     = matmul(Xt, X)
XtX_inv = inv2x2(XtX)
Xty     = matmul(Xt, Y_col)
beta    = matmul(XtX_inv, Xty)
print(beta)

import numpy as np

X_arr = np.array(X)          # [1, income] rows -> numpy array
y_arr = np.array(y)          # plain list of prices

beta = np.linalg.inv(X_arr.T @ X_arr) @ X_arr.T @ y_arr
print(beta)

[[43980.628156294], [42054.07487405656]]
[43980.6281563  42054.07487406]


In [15]:
# Here we make it adjustable for any number of predictors.
x2 = df['housing_median_age'].tolist()
x3 = df['total_rooms'].tolist()

X_multi = []
for i in range(len(x)):
    X_multi.append([1, x[i], x2[i], x3[i]])   # 1 + three predictors

# Make our predictors into a numpy list: an array.
X_multi = np.array(X_multi)

def fit_ols(X, y):
  X=np.array(X);
  y=np.array(y);
  return np.linalg.inv(X.T@X)@X.T@y

beta_multi = fit_ols(X_multi, y)
names = ["intercept", "income", "age", "rooms"]

# Let's make the output legible.
for i in range(len(names)):
    print(names[i], ":", round(beta_multi[i], 2))

# Now check against statsmodels params.
import statsmodels.api as sm
print(sm.OLS(np.array(y), X_multi).fit().params)

intercept : -24896.4
income : 42719.26
age : 1970.22
rooms : 3.77
[-2.48964021e+04  4.27192589e+04  1.97021766e+03  3.76995199e+00]
